# Project: FinFestAlpha
## PHASE 1: ENVIRONMENT SETUP & DATA SOURCING



In [2]:
# 1. Install required packages safely
!pip install yfinance pandas numpy --quiet
!pip install --upgrade yfinance


import os
import pandas as pd
import numpy as np
import yfinance as yf

from datetime import datetime

In [3]:
# 2. Establish local directory parameters inside Colab
DATA_DIR = "data1/raw"
os.makedirs(DATA_DIR, exist_ok=True)

#Mounting Drive
from google.colab import drive
drive.mount('/content/drive')
# change directory to save inside your actual Drive:
DATA_DIR = "/content/drive/MyDrive/data1/raw"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# 3. Define dates and active tickers
START_DATE = "2010-01-01"
END_DATE = datetime.today().strftime('%Y-%m-%d')

TICKER_MAP = {
    "NIFTY_50": "^NSEI",
    "NIFTY_FIN_SERVICES": "NIFTY_FIN_SERVICE.NS",  # Fixed with underscores
    "NIFTY_AUTO": "^CNXAUTO"
}

def fetch_finfest_alpha_data(tickers_dict, start, end):
    print(f"🚀 Initializing clean data extraction from {start} to {end}...")

    ticker_symbols = list(tickers_dict.values())
    # Download data with explicit auto_adjust parameters
    raw_download = yf.download(ticker_symbols, start=start, end=end, auto_adjust=True)

    # Modern yfinance returns MultiIndex structural columns: e.g., (Price, Ticker)
    # We want to isolate the 'Close' prices (which are pre-adjusted)
    if 'Close' in raw_download.columns.levels[0]:
      close_df = raw_download['Close']
    else:
      # Emergency structural fallback if index level sorting varies
      close_df = raw_download.xs('Close', axis=1, level=0)

    # Flip the human-readable naming map back to rename target headers safely
    reverse_map = {v: k for k, v in tickers_dict.items()}
    close_df = close_df.rename(columns=reverse_map)

    # Enforce standard column ordering sequence
    return close_df[list(tickers_dict.keys())]

    # Run the updated, crash-proof pipeline
    raw_market_data = fetch_festive_alpha_data(TICKER_MAP, START_DATE, END_DATE)





# DATA QUALITY ASSURANCE (QA) DIAGNOSTICS


In [5]:
raw_market_data = fetch_finfest_alpha_data(TICKER_MAP, START_DATE, END_DATE)

print("\n🔍 Running Data Health Diagnostics...")
print("-" * 45)
print(f"Total Trading Days Captured: {len(raw_market_data)}")

# The index is now successfully recognized as a true DatetimeIndex
print(f"Temporal Boundaries: {raw_market_data.index.min().date()} to {raw_market_data.index.max().date()}")

# Count tracking gaps
null_counts = raw_market_data.isnull().sum()
print("\nMissing Data Check Matrix:")
print(null_counts)

# Bridge any minor national market holiday tracking discrepancies
if null_counts.sum() > 0:
    print("\n🩹 Gaps detected. Applying forward-fill (FFILL) to preserve timeline continuity...")
    raw_market_data = raw_market_data.ffill().bfill()

# Save the final structured file to your workspace
output_file = os.path.join(DATA_DIR, "raw_market_indices.csv")

# CRITICAL FIX: Create the folders inside your Drive if they don't exist yet
os.makedirs(DATA_DIR, exist_ok=True)

# Now Pandas will find the directory and save successfully!
raw_market_data.to_csv(output_file)
print(f"\n💾 Phase 1 complete. Extracted data saved inside cloud workspace at: {output_file}")

🚀 Initializing clean data extraction from 2010-01-01 to 2026-06-08...


[*********************100%***********************]  3 of 3 completed


🔍 Running Data Health Diagnostics...
---------------------------------------------
Total Trading Days Captured: 4047
Temporal Boundaries: 2010-01-04 to 2026-06-05

Missing Data Check Matrix:
Ticker
NIFTY_50               15
NIFTY_FIN_SERVICES    435
NIFTY_AUTO            396
dtype: int64

🩹 Gaps detected. Applying forward-fill (FFILL) to preserve timeline continuity...

💾 Phase 1 complete. Extracted data saved inside cloud workspace at: /content/drive/MyDrive/data1/raw/raw_market_indices.csv
